# LeetCode 1007 — Minimum Domino Rotations For Equal Row bằng Python

Link đề: https://leetcode.com/problems/minimum-domino-rotations-for-equal-row/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Greedy (tham lam)

Skeleton LeetCode:

```python
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        ...
```

Ghi chú: có một hàng các quân domino, quân thứ `i` có số `tops[i]` ở nửa trên và `bottoms[i]` ở nửa dưới (mỗi nửa là một số từ `1` đến `6`). Ta được **xoay** quân thứ `i` để hai số hoán đổi. Tìm **số lần xoay ít nhất** để **mọi số ở hàng trên bằng nhau**, hoặc **mọi số ở hàng dưới bằng nhau**; không làm được thì trả về `-1`. Ý then chốt: nếu có cách làm hàng nào đó đều bằng giá trị `v` thì **mọi quân** đều phải chứa `v` ở ít nhất một nửa, nên `v` buộc phải là `tops[0]` **hoặc** `bottoms[0]` (quân đầu tiên): chỉ có **hai** giá trị ứng viên cần thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai hàng, thao tác xoay từng quân, mục tiêu "một trong hai hàng đồng nhất", đáp án nhỏ nhất hoặc `-1`.
2. Nắm các công cụ Python cần thiết: `zip` hai danh sách, `min` với giá trị sentinel, bộ đếm `[0] * 7`, `collections.Counter`, và duyệt mặt nạ bit `range(1 << n)` cho vét cạn.
3. Hiểu nguyên lý: giá trị đích `v` phải có trên **mọi** quân (ít nhất một nửa), từ đó chỉ ứng viên là `tops[0]` hoặc `bottoms[0]`; với mỗi ứng viên, số lần xoay để hàng trên toàn `v` là số quân có `tops[i] != v` (khi đó `bottoms[i] == v` bắt buộc), và tương tự cho hàng dưới.
4. Đi qua 4 cách: thử mọi tập các quân đem xoay (brute force, oracle cho `n` nhỏ), thử cả 6 giá trị, chỉ thử hai ứng viên (cách nộp chính), và đếm tần suất một lượt.
5. Tránh các lỗi kinh điển: chỉ xét một hàng, quên kiểm tra khả thi, đếm sai khi hai nửa cùng bằng `v`, chỉ thử ứng viên `tops[0]`, và trả về `0` hoặc số vô cực thay vì `-1`.

## 2. Đề bài & ví dụ

Trong một hàng domino, `tops[i]` và `bottoms[i]` là nửa trên và nửa dưới của quân thứ `i`. (Một quân domino là một viên có hai số từ `1` đến `6`, mỗi nửa một số.)

Ta có thể xoay quân thứ `i`, khi đó `tops[i]` và `bottoms[i]` đổi chỗ cho nhau.

Trả về **số lần xoay ít nhất** để **mọi số ở `tops` bằng nhau**, hoặc **mọi số ở `bottoms` bằng nhau**. Nếu không thể, trả về `-1`.

**Ví dụ 1:** `tops = [2,1,2,4,2,2]`, `bottoms = [5,2,6,2,3,2]` -> `2`. Xoay quân thứ 2 và thứ 4 (chỉ số 1 và 3) thì hàng trên toàn `2`.

**Ví dụ 2:** `tops = [3,5,1,2,3]`, `bottoms = [3,6,3,3,4]` -> `-1`. Không thể làm một hàng đồng nhất.

**Ràng buộc:**

- `2 <= tops.length <= 2 * 10^4`
- `bottoms.length == tops.length`
- `1 <= tops[i], bottoms[i] <= 6`

In [1]:
from typing import List, Optional
from collections import Counter
import random, time

examples = [
    ([2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2], 2),
    ([3, 5, 1, 2, 3], [3, 6, 3, 3, 4], -1),
]
for tops, bottoms, expected in examples:
    print(f"tops={tops} bottoms={bottoms} -> kỳ vọng {expected}")

tops=[2, 1, 2, 4, 2, 2] bottoms=[5, 2, 6, 2, 3, 2] -> kỳ vọng 2
tops=[3, 5, 1, 2, 3] bottoms=[3, 6, 3, 3, 4] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Xoay một quân và "hàng đồng nhất"

Xoay quân `i` hoán đổi `tops[i]` và `bottoms[i]`; hai nửa của một quân luôn "đi cùng nhau", **tổng hợp tập hai số** của quân không đổi. Hàng trên đồng nhất với giá trị `v` ⇔ mọi quân có `v` ở nửa trên sau khi xoay ⇔ mọi quân **chứa `v` ở ít nhất một nửa**.

In [2]:
tops, bottoms = [2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2]
v = 2
print("quân nào chứa", v, "?", [(i, (t, b)) for i, (t, b) in enumerate(zip(tops, bottoms)) if v in (t, b)])
print("quân phải xoay để hàng trên toàn", v, ":", [i for i, (t, b) in enumerate(zip(tops, bottoms)) if t != v and b == v])

quân nào chứa 2 ? [(0, (2, 5)), (1, (1, 2)), (2, (2, 6)), (3, (4, 2)), (4, (2, 3)), (5, (2, 2))]
quân phải xoay để hàng trên toàn 2 : [1, 3]


### 3.2. Hai ứng viên

Nếu có cách làm một hàng toàn `v` thì **quân đầu tiên** phải chứa `v`, tức `v == tops[0]` hoặc `v == bottoms[0]`. Vậy ta chỉ cần thử hai giá trị (nếu bằng nhau thì chỉ một). Không cần thử cả 6 giá trị.

### 3.3. Đếm số lần xoay cho một giá trị đích

Với giá trị `v` mà mọi quân đều chứa `v`: để hàng trên toàn `v` ta xoay các quân có `tops[i] != v` (`bottoms[i] == v` chắc chắn); để hàng dưới toàn `v` ta xoay các quân có `bottoms[i] != v`. Đáp án cho `v` là số nhỏ hơn trong hai số đó.

In [3]:
def rotations_for(tops, bottoms, v):
    rt = rb = 0
    for t, b in zip(tops, bottoms):
        if t != v and b != v:
            return -1
        if t != v:
            rt += 1                 # phải xoay để đưa v lên hàng trên
        elif b != v:
            rb += 1                 # phải xoay để đưa v xuống hàng dưới
    return min(rt, rb)

print("v = 2:", rotations_for([2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2], 2), "| v = 5:", rotations_for([2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2], 5))

v = 2: 2 | v = 5: -1


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `tops = [2,1,2,4,2,2]`, `bottoms = [5,2,6,2,3,2]`.

- Quân đầu: `(2, 5)`: ứng viên `v = 2` hoặc `v = 5`.
- `v = 5`: quân thứ hai `(1, 2)` không chứa `5` ⇒ không khả thi.
- `v = 2`: mọi quân đều chứa `2` (`(2,5)`, `(1,2)`, `(2,6)`, `(4,2)`, `(2,3)`, `(2,2)`). Hàng trên cần xoay các quân có `tops != 2`: chỉ số `1` và `3`: `2` lần. Hàng dưới cần xoay các quân có `bottoms != 2`: chỉ số `0, 2, 4`: `3` lần. Nhỏ nhất `2`.

Ví dụ 2: quân đầu `(3, 3)`: ứng viên chỉ `3`. Quân thứ hai `(5, 6)` không chứa `3` ⇒ `-1`.

## 5. Cách 1 — Brute Force: thử mọi tập quân đem xoay

**Ý tưởng.** Với `n` quân có `2^n` cách chọn tập quân xoay. Với mỗi tập (một mặt nạ bit), dựng hai hàng sau khi xoay và kiểm tra hàng trên có đồng nhất hoặc hàng dưới có đồng nhất; nếu có thì ghi nhận số quân đã xoay (số bit `1`); lấy nhỏ nhất. Đây là **định nghĩa trực tiếp** của đề nên là **oracle độc lập** (không dùng giả thiết "ứng viên là quân đầu").

In [4]:
def domino_brute(tops: List[int], bottoms: List[int]) -> int:
    n = len(tops)
    best = None
    for mask in range(1 << n):
        top, bottom = [], []
        for i in range(n):
            if mask >> i & 1:
                top.append(bottoms[i]); bottom.append(tops[i])
            else:
                top.append(tops[i]); bottom.append(bottoms[i])
        if len(set(top)) == 1 or len(set(bottom)) == 1:
            cnt = bin(mask).count("1")
            if best is None or cnt < best:
                best = cnt
    return -1 if best is None else best

for tops, bottoms, expected in examples:
    got = domino_brute(tops, bottoms)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 2 OK
brute = -1 OK


**Độ phức tạp và nút thắt.** `O(2^n * n)`: với `n = 2 * 10^4` là không thể. Đo thử:

In [5]:
rng = random.Random(1007)
for n in (8, 12, 14, 16):
    tops = [rng.randint(1, 3) for _ in range(n)]
    bottoms = [rng.randint(1, 3) for _ in range(n)]
    t0 = time.perf_counter()
    domino_brute(tops, bottoms)
    print(f"n={n:2}: {(time.perf_counter() - t0) * 1000:9.1f} ms")
print("n = 20000: không thể")

n= 8:       0.5 ms
n=12:       9.2 ms
n=14:      41.6 ms


n=16:     183.8 ms
n = 20000: không thể


Nút thắt: ta xét mọi tập quân xoay, trong khi **giá trị đích** chỉ có tối đa `6` khả năng (thậm chí `2`) và với mỗi giá trị đích tập quân cần xoay được **xác định duy nhất**.

## 6. Ý tưởng thuật toán / bước đột phá

**Chọn giá trị đích `v`; số lần xoay được xác định.** Với `v` cố định và hàng đích là hàng trên: quân `i` cần xoay ⇔ `tops[i] != v` (và phải có `bottoms[i] == v`). Không còn lựa chọn nào: ta chỉ cần **đếm**. Chỉ còn câu hỏi thử những `v` nào:

- thử cả `6` giá trị (`O(6n)`), hoặc
- chỉ thử `tops[0]` và `bottoms[0]` (`O(2n)`), vì `v` phải xuất hiện trên quân đầu tiên.

| Cách | Ý tưởng |
|---|---|
| 2 | Thử mọi `v` từ `1` đến `6` |
| 3 | Chỉ thử `tops[0]` và `bottoms[0]` (nộp) |
| 4 | Đếm một lượt: `v` khả thi ⇔ `count_top + count_bottom - count_both == n`, số xoay `n - max(count_top, count_bottom)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi giá trị `v` từ `1` đến `6`

**Giải thích từng đoạn.** Với mỗi `v`, gọi `rotations_for` (mục 3.3): trả về `-1` nếu có quân không chứa `v`, ngược lại số lần xoay nhỏ nhất cho `v` (hàng trên hoặc hàng dưới). Đáp án là nhỏ nhất trong các kết quả khác `-1`.

In [6]:
def domino_all_values(tops: List[int], bottoms: List[int]) -> int:
    best = -1
    for v in range(1, 7):
        r = rotations_for(tops, bottoms, v)
        if r != -1 and (best == -1 or r < best):
            best = r
    return best

for tops, bottoms, expected in examples:
    got = domino_all_values(tops, bottoms)
    print(f"thử 6 giá trị = {got}", "OK" if got == expected else "SAI")
    assert got == expected

thử 6 giá trị = 2 OK
thử 6 giá trị = -1 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        def check(v: int) -> int:
            rt = rb = 0
            for t, b in zip(tops, bottoms):
                if t != v and b != v:
                    return -1                  # quân này không chứa v
                if t != v:
                    rt += 1                    # xoay để đưa v lên hàng trên
                elif b != v:
                    rb += 1                    # xoay để đưa v xuống hàng dưới
            return min(rt, rb)

        best = -1
        for v in range(1, 7):
            r = check(v)
            if r != -1 and (best == -1 or r < best):
                best = r
        return best

sol_all = Solution().minDominoRotations
for tops, bottoms, expected in examples:
    assert sol_all(tops, bottoms) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(6n)`, bộ nhớ `O(1)`.

---

### 7.2. Cách 3 — Chỉ thử hai ứng viên (cách nộp chính)

**Ý tưởng khác.** Quân đầu tiên `(tops[0], bottoms[0])` phải chứa giá trị đích, nên chỉ cần thử `v = tops[0]` và `v = bottoms[0]`. Gọi `check(v)` như trên, lấy nhỏ nhất trong các kết quả không phải `-1`.

**Giải thích từng đoạn.**

- `check(v)` trả về `-1` nếu có quân không chứa `v`; ngược lại số lần xoay nhỏ nhất.
- Gom kết quả của hai ứng viên; nếu cả hai `-1` thì `-1`.

In [8]:
def domino_two_candidates(tops: List[int], bottoms: List[int]) -> int:
    results = [rotations_for(tops, bottoms, v) for v in {tops[0], bottoms[0]}]
    ok = [r for r in results if r != -1]
    return min(ok) if ok else -1

for tops, bottoms, expected in examples:
    got = domino_two_candidates(tops, bottoms)
    print(f"hai ứng viên = {got}", "OK" if got == expected else "SAI")
    assert got == expected

hai ứng viên = 2 OK
hai ứng viên = -1 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        def check(v: int) -> int:
            rt = rb = 0
            for t, b in zip(tops, bottoms):
                if t != v and b != v:
                    return -1                  # quân này không chứa v: v không thể là giá trị đích
                if t != v:
                    rt += 1                    # phải xoay để đưa v lên hàng trên
                elif b != v:
                    rb += 1                    # phải xoay để đưa v xuống hàng dưới
            return min(rt, rb)

        ans = [r for r in (check(tops[0]), check(bottoms[0])) if r != -1]   # chỉ hai ứng viên
        return min(ans) if ans else -1

sol_two = Solution().minDominoRotations
for tops, bottoms, expected in examples:
    assert sol_two(tops, bottoms) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(2n)` (hai lượt duyệt), bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Đếm tần suất một lượt

**Ý tưởng khác.** Đếm `ct[v]` = số quân có `tops[i] == v`, `cb[v]` = số quân có `bottoms[i] == v`, và `both[v]` = số quân có **cả hai nửa** bằng `v`. Số quân chứa `v` ở **ít nhất một nửa** là `ct[v] + cb[v] - both[v]` (nguyên lý bù trừ); `v` khả thi ⇔ giá trị này bằng `n`. Khi khả thi: số lần xoay để hàng trên toàn `v` là `n - ct[v]`, hàng dưới là `n - cb[v]`; nhỏ nhất `n - max(ct[v], cb[v])`.

**Giải thích từng đoạn.** Một lượt duyệt điền ba mảng đếm; sau đó thử `6` giá trị trong `O(6)`.

In [10]:
def domino_counting(tops: List[int], bottoms: List[int]) -> int:
    n = len(tops)
    ct, cb, both = [0] * 7, [0] * 7, [0] * 7
    for t, b in zip(tops, bottoms):
        ct[t] += 1
        cb[b] += 1
        if t == b:
            both[t] += 1
    best = -1
    for v in range(1, 7):
        if ct[v] + cb[v] - both[v] == n:               # mọi quân chứa v
            r = n - max(ct[v], cb[v])
            if best == -1 or r < best:
                best = r
    return best

for tops, bottoms, expected in examples:
    got = domino_counting(tops, bottoms)
    print(f"đếm tần suất = {got}", "OK" if got == expected else "SAI")
    assert got == expected

đếm tần suất = 2 OK
đếm tần suất = -1 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        n = len(tops)
        ct, cb, both = [0] * 7, [0] * 7, [0] * 7       # đếm ở hàng trên, hàng dưới, và quân hai nửa cùng giá trị
        for t, b in zip(tops, bottoms):
            ct[t] += 1
            cb[b] += 1
            if t == b:
                both[t] += 1
        best = -1
        for v in range(1, 7):
            if ct[v] + cb[v] - both[v] == n:           # mọi quân chứa v ở ít nhất một nửa
                r = n - max(ct[v], cb[v])              # xoay những quân chưa có v ở hàng đích
                if best == -1 or r < best:
                    best = r
        return best

sol_count = Solution().minDominoRotations
for tops, bottoms, expected in examples:
    assert sol_count(tops, bottoms) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n + 6)` (một lượt duyệt), bộ nhớ `O(1)` (ba mảng độ dài `7`).

## 8. TRACE từng bước

### 8.1. Cách 3 cho ví dụ 1

In kiểm tra từng quân cho mỗi ứng viên.

In [12]:
def trace_check(tops, bottoms, v):
    rt = rb = 0
    print(f"--- thử v = {v}")
    for i, (t, b) in enumerate(zip(tops, bottoms)):
        if t != v and b != v:
            print(f"quân {i} = ({t},{b}): không chứa {v} -> không khả thi")
            return -1
        if t != v:
            rt += 1
            print(f"quân {i} = ({t},{b}): hàng trên thiếu {v} -> xoay (rt = {rt})")
        elif b != v:
            rb += 1
            print(f"quân {i} = ({t},{b}): hàng dưới thiếu {v} -> xoay để làm hàng dưới (rb = {rb})")
        else:
            print(f"quân {i} = ({t},{b}): cả hai nửa đều {v}")
    print(f"hàng trên cần {rt} lần, hàng dưới cần {rb} lần -> {min(rt, rb)}")
    return min(rt, rb)

tops, bottoms = [2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2]
r2, r5 = trace_check(tops, bottoms, tops[0]), trace_check(tops, bottoms, bottoms[0])
print("kết quả:", r2, r5)
assert (r2, r5) == (2, -1)

--- thử v = 2
quân 0 = (2,5): hàng dưới thiếu 2 -> xoay để làm hàng dưới (rb = 1)
quân 1 = (1,2): hàng trên thiếu 2 -> xoay (rt = 1)
quân 2 = (2,6): hàng dưới thiếu 2 -> xoay để làm hàng dưới (rb = 2)
quân 3 = (4,2): hàng trên thiếu 2 -> xoay (rt = 2)
quân 4 = (2,3): hàng dưới thiếu 2 -> xoay để làm hàng dưới (rb = 3)
quân 5 = (2,2): cả hai nửa đều 2
hàng trên cần 2 lần, hàng dưới cần 3 lần -> 2
--- thử v = 5
quân 0 = (2,5): hàng trên thiếu 5 -> xoay (rt = 1)
quân 1 = (1,2): không chứa 5 -> không khả thi
kết quả: 2 -1


### 8.2. Cách 4 cho ví dụ 1: các mảng đếm

In `ct`, `cb`, `both` và điều kiện khả thi cho từng `v`.

In [13]:
tops, bottoms = [2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2]
n = len(tops)
ct, cb, both = [0] * 7, [0] * 7, [0] * 7
for t, b in zip(tops, bottoms):
    ct[t] += 1; cb[b] += 1
    if t == b:
        both[t] += 1
print("ct   =", ct[1:], "\ncb   =", cb[1:], "\nboth =", both[1:])
for v in range(1, 7):
    covered = ct[v] + cb[v] - both[v]
    print(f"v={v}: quân chứa v = {covered}/{n}", "-> khả thi, xoay", n - max(ct[v], cb[v]) if covered == n else "-> không")

ct   = [1, 4, 0, 1, 0, 0] 
cb   = [0, 3, 1, 0, 1, 1] 
both = [0, 1, 0, 0, 0, 0]
v=1: quân chứa v = 1/6 -> khả thi, xoay -> không
v=2: quân chứa v = 6/6 -> khả thi, xoay 2
v=3: quân chứa v = 1/6 -> khả thi, xoay -> không
v=4: quân chứa v = 1/6 -> khả thi, xoay -> không
v=5: quân chứa v = 1/6 -> khả thi, xoay -> không
v=6: quân chứa v = 1/6 -> khả thi, xoay -> không


### 8.3. Trường hợp hai ứng viên khác nhau cho đáp án khác nhau

`tops = [1, 2, 2, 1, 1]`, `bottoms = [2, 1, 1, 2, 2]`: ứng viên `1` và `2` đều khả thi, nhưng số lần xoay khác nhau tùy hàng đích; ta lấy nhỏ nhất.

In [14]:
tops, bottoms = [1, 2, 2, 1, 1], [2, 1, 1, 2, 2]
print("v=1:", rotations_for(tops, bottoms, 1), "| v=2:", rotations_for(tops, bottoms, 2), "| đáp án:", domino_two_candidates(tops, bottoms), "| brute:", domino_brute(tops, bottoms))
assert domino_two_candidates(tops, bottoms) == domino_brute(tops, bottoms) == 2

v=1: 2 | v=2: 2 | đáp án: 2 | brute: 2


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (chỉ thử hai ứng viên): ngắn, `O(n)` và dễ chứng minh đúng.

```python
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        def check(v: int) -> int:
            rt = rb = 0
            for t, b in zip(tops, bottoms):
                if t != v and b != v:
                    return -1                  # quân này không chứa v: v không thể là giá trị đích
                if t != v:
                    rt += 1                    # phải xoay để đưa v lên hàng trên
                elif b != v:
                    rb += 1                    # phải xoay để đưa v xuống hàng dưới
            return min(rt, rb)

        ans = [r for r in (check(tops[0]), check(bottoms[0])) if r != -1]   # chỉ hai ứng viên
        return min(ans) if ans else -1
```

In [15]:
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        def check(v: int) -> int:
            rt = rb = 0
            for t, b in zip(tops, bottoms):
                if t != v and b != v:
                    return -1                  # quân này không chứa v: v không thể là giá trị đích
                if t != v:
                    rt += 1                    # phải xoay để đưa v lên hàng trên
                elif b != v:
                    rb += 1                    # phải xoay để đưa v xuống hàng dưới
            return min(rt, rb)

        ans = [r for r in (check(tops[0]), check(bottoms[0])) if r != -1]   # chỉ hai ứng viên
        return min(ans) if ans else -1

sol = Solution()
for tops, bottoms, expected in examples:
    got = sol.minDominoRotations(tops, bottoms)
    print(f"{got}", "OK" if got == expected else "SAI")
    assert got == expected

2 OK
-1 OK


## 10. Vì sao đúng

**Ứng viên.** Nếu sau khi xoay hàng trên (hoặc hàng dưới) toàn `v`, thì quân đầu tiên, sau khi xoay hoặc không, có `v` ở hàng đó, nên `v` thuộc `{tops[0], bottoms[0]}`. Vậy chỉ cần thử hai giá trị.

**Với `v` cố định, hàng đích (trên).** Hàng trên toàn `v` ⇔ mỗi quân có `v` ở nửa trên sau khi xoay. Quân `i` có `tops[i] == v`: không cần xoay (xoay chỉ tốn thêm và có thể đẩy `v` xuống). Quân `i` có `tops[i] != v`: **bắt buộc** xoay, và điều đó khả thi ⇔ `bottoms[i] == v`. Vậy số lần xoay tối thiểu là `#{i : tops[i] != v}` nếu mọi quân chứa `v`, ngược lại không khả thi. Hàng dưới đối xứng: `#{i : bottoms[i] != v}`.

**Gom lại.** Đáp án là nhỏ nhất trên mọi `v` khả thi và mọi hàng đích. Trong `check(v)` ta đếm đồng thời: `rt` = số quân cần xoay cho hàng trên (`tops != v`, nên `bottoms == v`), `rb` = số quân cần xoay cho hàng dưới (`bottoms != v` khi `tops == v`). Quân có `tops == bottoms == v` không cần xoay cho cả hai hàng. Đếm `rt` ở nhánh `t != v` và `rb` ở nhánh `b != v` (khi `t == v`) đúng bằng hai số đó.

**Cách 4.** Số quân chứa `v` là `ct[v] + cb[v] - both[v]` (bù trừ); `rt = n - ct[v]` và `rb = n - cb[v]`.

**Độ phức tạp.** `O(n)`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) thử **mọi** tập quân xoay theo định nghĩa. Ta đối chiếu bốn cách trên **mọi** cấu hình `n <= 3` với giá trị `1..6` và `n = 4` với giá trị `1..3`, rồi trên nhiều cấu hình ngẫu nhiên có seed (cả cấu hình cài sẵn để chắc chắn khả thi).

In [16]:
from itertools import product
total = feasible = 0
for n, vmax in ((2, 6), (3, 6), (4, 3)):
    for cells in product(range(1, vmax + 1), repeat=2 * n):
        tops, bottoms = list(cells[:n]), list(cells[n:])
        ref = domino_brute(tops, bottoms)
        assert domino_all_values(tops, bottoms) == domino_two_candidates(tops, bottoms) == domino_counting(tops, bottoms) == ref, (tops, bottoms)
        total += 1
        feasible += ref != -1
print(f"đã đối chiếu {total} cấu hình, trong đó {feasible} khả thi, {total - feasible} cho -1")

đã đối chiếu 54513 cấu hình, trong đó 10359 khả thi, 44154 cho -1


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập quân xoay | `O(2^n * n)` | `O(n)` |
| 2. Thử 6 giá trị | `O(6n)` | `O(1)` |
| 3. Thử 2 ứng viên | `O(2n)` | `O(1)` |
| 4. Đếm tần suất | `O(n)` | `O(1)` |

Đo ở cỡ tối đa (`n = 2 * 10^4`), với cấu hình khả thi và không khả thi:

In [17]:
def planted_case(rng, n, target, noise_p=0.0):
    tops, bottoms = [], []
    for _ in range(n):
        other = rng.randint(1, 6)
        if rng.random() < 0.5:
            t, b = target, other
        else:
            t, b = other, target
        tops.append(t); bottoms.append(b)
    if noise_p and rng.random() < noise_p:
        i = rng.randrange(n)
        tops[i], bottoms[i] = (target % 6) + 1, (target % 6) + 1      # làm hỏng một quân
    return tops, bottoms

rng = random.Random(11)
n = 2 * 10 ** 4
cases = {"khả thi (cài sẵn)": planted_case(rng, n, 3),
         "không khả thi (bị hỏng)": planted_case(rng, n, 3, noise_p=1.0),
         "ngẫu nhiên": ([rng.randint(1, 6) for _ in range(n)], [rng.randint(1, 6) for _ in range(n)])}
for label, (tops, bottoms) in cases.items():
    ref = domino_counting(tops, bottoms)
    for name, fn in (("Cách 2", sol_all), ("Cách 3", sol_two), ("Cách 4", sol_count), ("nộp", sol.minDominoRotations)):
        t0 = time.perf_counter()
        got = fn(tops, bottoms)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, name
        print(f"{label:26} {name:7}: {dt:7.2f} ms, đáp án {got}")

khả thi (cài sẵn)          Cách 2 :    1.95 ms, đáp án 8272
khả thi (cài sẵn)          Cách 3 :    1.99 ms, đáp án 8272
khả thi (cài sẵn)          Cách 4 :    2.80 ms, đáp án 8272
khả thi (cài sẵn)          nộp    :    2.18 ms, đáp án 8272
không khả thi (bị hỏng)    Cách 2 :    0.61 ms, đáp án -1
không khả thi (bị hỏng)    Cách 3 :    0.58 ms, đáp án -1
không khả thi (bị hỏng)    Cách 4 :    2.82 ms, đáp án -1
không khả thi (bị hỏng)    nộp    :    0.64 ms, đáp án -1
ngẫu nhiên                 Cách 2 :    0.01 ms, đáp án -1
ngẫu nhiên                 Cách 3 :    0.00 ms, đáp án -1
ngẫu nhiên                 Cách 4 :    2.93 ms, đáp án -1
ngẫu nhiên                 nộp    :    0.01 ms, đáp án -1


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc, cả cỡ tối đa) và các tính chất.

In [18]:
def all_solutions():
    return {"all_values": domino_all_values, "two": domino_two_candidates, "counting": domino_counting,
            "sol_all": sol_all, "sol_two": sol_two, "sol_count": sol_count, "final": sol.minDominoRotations}

def check_all(tops, bottoms, expected=None):
    results = {name: fn(list(tops), list(bottoms)) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (tops[:10], bottoms[:10], results)
    ref = results["two"]
    if len(tops) <= 14:
        assert domino_brute(tops, bottoms) == ref, (tops, bottoms, ref, domino_brute(tops, bottoms))
    if expected is not None:
        assert ref == expected, (tops, bottoms, ref, expected)
    return ref

# 12.1 ví dụ đề
for tops, bottoms, expected in examples:
    check_all(tops, bottoms, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1, 2], [2, 1], 1),                         # n = 2: xoay một quân để hàng trên toàn 1 (hoặc toàn 2)
    ([1, 1], [1, 1], 0),                         # đã đồng nhất cả hai hàng
    ([1, 1], [2, 2], 0),                         # hàng trên đã toàn 1
    ([1, 2], [3, 4], -1),                        # hai quân không chung giá trị
    ([1, 2, 3], [4, 5, 6], -1),                  # không có giá trị chung
    ([2, 2, 2, 2], [1, 2, 3, 4], 0),             # hàng trên đã toàn 2
    ([1, 2, 2, 2], [2, 1, 3, 4], 1),             # xoay quân đầu: hàng trên [2,2,2,2]
    ([1, 2, 3, 1], [2, 1, 1, 3], 2),             # v = 1: hàng trên [1,2,3,1] cần xoay 2 quân (chỉ số 1, 2) ; hàng dưới cần xoay 2 quân
    ([3, 3, 3, 1, 3], [1, 1, 1, 3, 1], 1),       # hàng trên toàn 3 bằng cách xoay quân thứ 4 (chỉ số 3)
    ([1, 5, 1, 1, 1], [5, 1, 5, 5, 5], 1),       # tops thiếu 1 ở chỉ số 1; bottoms thiếu 5 ở chỉ số 1: xoay một quân
    ([1] * 6, [1] * 6, 0),                       # toàn bộ giống nhau
    ([6, 6, 6], [6, 1, 2], 0),                   # hàng trên đã toàn 6; bottoms có hai quân khác 6
    ([1, 2, 1, 2], [2, 1, 2, 1], 2),             # v = 1: hàng trên cần xoay 2 quân; v = 2: hàng trên cần xoay 2 quân
]
for tops, bottoms, expected in edge:
    check_all(tops, bottoms, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed so với oracle vét cạn)
def wrong_top_only(tops, bottoms):               # chỉ xét làm hàng trên đồng nhất
    best = -1
    for v in range(1, 7):
        cnt = 0
        ok = True
        for t, b in zip(tops, bottoms):
            if t != v and b != v:
                ok = False
                break
            if t != v:
                cnt += 1
        if ok and (best == -1 or cnt < best):
            best = cnt
    return best

def wrong_no_feasibility(tops, bottoms):         # đếm quân có tops != v mà không kiểm tra bottoms == v
    best = None
    for v in {tops[0], bottoms[0]}:
        rt = sum(1 for t in tops if t != v)
        rb = sum(1 for b in bottoms if b != v)
        c = min(rt, rb)
        if best is None or c < best:
            best = c
    return best

def wrong_first_candidate_only(tops, bottoms):   # chỉ thử ứng viên tops[0]
    v = tops[0]
    rt = rb = 0
    for t, b in zip(tops, bottoms):
        if t != v and b != v:
            return -1
        if t != v:
            rt += 1
        elif b != v:
            rb += 1
    return min(rt, rb)

def wrong_double_count(tops, bottoms):           # đếm cả hai nhánh độc lập: coi quân (v, v) là phải xoay cho hàng dưới
    best = -1
    for v in {tops[0], bottoms[0]}:
        rt = rb = 0
        ok = True
        for t, b in zip(tops, bottoms):
            if t != v and b != v:
                ok = False
                break
            rt += t != v
            rb += b != v
            if t == v and b == v:
                rt += 1                          # sai: quân (v, v) không cần xoay
        if ok:
            c = min(rt, rb)
            if best == -1 or c < best:
                best = c
    return best

def wrong_zero_when_impossible(tops, bottoms):   # trả 0 thay vì -1 khi không khả thi
    r = domino_two_candidates(tops, bottoms)
    return 0 if r == -1 else r

def find_counter(wrong, vmax=3, max_n=5):
    for n in range(2, max_n + 1):
        for cells in product(range(1, vmax + 1), repeat=2 * n):
            tops, bottoms = list(cells[:n]), list(cells[n:])
            if wrong(list(tops), list(bottoms)) != domino_brute(tops, bottoms):
                return tops, bottoms
    return None

counters = {}
for name, fn in (("chỉ làm hàng trên đồng nhất", wrong_top_only), ("quên kiểm tra khả thi", wrong_no_feasibility),
                 ("chỉ thử ứng viên tops[0]", wrong_first_candidate_only), ("đếm sai quân hai nửa cùng giá trị", wrong_double_count),
                 ("trả 0 thay vì -1", wrong_zero_when_impossible)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    tops, bottoms = case
    print(f"{name:36} tops={tops} bottoms={bottoms}: sai -> {fn(list(tops), list(bottoms))}, đúng -> {domino_brute(tops, bottoms)}")
    check_all(tops, bottoms)
print("12.3 phản ví dụ: OK")

chỉ làm hàng trên đồng nhất          tops=[1, 2] bottoms=[1, 1]: sai -> 1, đúng -> 0
quên kiểm tra khả thi                tops=[1, 2] bottoms=[1, 2]: sai -> 1, đúng -> -1
chỉ thử ứng viên tops[0]             tops=[1, 2] bottoms=[2, 2]: sai -> -1, đúng -> 0
đếm sai quân hai nửa cùng giá trị    tops=[1, 1] bottoms=[1, 2]: sai -> 1, đúng -> 0
trả 0 thay vì -1                     tops=[1, 2] bottoms=[1, 2]: sai -> 0, đúng -> -1
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ so với oracle (n <= 14), nhiều mức khó khả thi: giá trị 1..2 (dễ khả thi), 1..3, 1..6 (hay -1)
rng = random.Random(4007)
feasible = infeasible = 0
for _ in range(1500):
    n = rng.randint(2, 12)
    vmax = rng.choice([2, 3, 6])
    tops = [rng.randint(1, vmax) for _ in range(n)]
    bottoms = [rng.randint(1, vmax) for _ in range(n)]
    ref = check_all(tops, bottoms)
    feasible += ref != -1
    infeasible += ref == -1
print(f"12.4 ngẫu nhiên nhỏ: {feasible} khả thi, {infeasible} không khả thi: OK")
assert feasible > 100 and infeasible > 100

# 12.5 cài sẵn khả thi (có đáp án <= số quân không chứa v ở hàng đích) và cỡ lớn n = 2 * 10^4
rng = random.Random(5007)
for _ in range(300):
    n = rng.randint(2, 14)
    target = rng.randint(1, 6)
    tops, bottoms = planted_case(rng, n, target)
    ref = check_all(tops, bottoms)
    assert ref != -1 and ref <= n // 2                                    # xoay tối đa n/2 quân (chọn hàng có v nhiều hơn)
for _ in range(10):
    n = rng.randint(15000, 20000)
    tops, bottoms = planted_case(rng, n, rng.randint(1, 6), noise_p=rng.choice([0.0, 1.0]))
    check_all(tops, bottoms)
print("12.5 cài sẵn và cỡ lớn (n tới 2 * 10^4): OK")

# 12.6 tính chất: hoán đổi hai hàng không đổi đáp án; đảo thứ tự quân không đổi; đáp án <= n // 2 khi khả thi; xoay trước một quân không đổi tính khả thi và làm đáp án lệch tối đa 1
rng = random.Random(6007)
for _ in range(500):
    n = rng.randint(2, 12)
    tops = [rng.randint(1, 3) for _ in range(n)]
    bottoms = [rng.randint(1, 3) for _ in range(n)]
    a = sol.minDominoRotations(tops, bottoms)
    assert sol.minDominoRotations(bottoms, tops) == a
    order = list(range(n)); rng.shuffle(order)
    assert sol.minDominoRotations([tops[i] for i in order], [bottoms[i] for i in order]) == a
    if a != -1:
        assert a <= n // 2
    i = rng.randrange(n)
    t2, b2 = list(tops), list(bottoms)
    t2[i], b2[i] = b2[i], t2[i]
    a2 = sol.minDominoRotations(t2, b2)
    assert (a2 == -1) == (a == -1)                                        # xoay trước một quân không đổi tính khả thi
    if a != -1:
        assert abs(a2 - a) <= 1                                           # nhưng số lần xoay còn lại lệch tối đa 1
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: 324 khả thi, 1176 không khả thi: OK


12.5 cài sẵn và cỡ lớn (n tới 2 * 10^4): OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét làm hàng trên đồng nhất

Đề cho phép làm **hàng trên hoặc hàng dưới** đồng nhất; quên hàng dưới có thể bỏ lỡ đáp án nhỏ hơn.

In [21]:
tops, bottoms = counters["chỉ làm hàng trên đồng nhất"]
print("tops =", tops, "bottoms =", bottoms, "| sai ->", wrong_top_only(list(tops), list(bottoms)), "| đúng ->", Solution().minDominoRotations(tops, bottoms))
assert wrong_top_only(list(tops), list(bottoms)) != Solution().minDominoRotations(tops, bottoms)

tops = [1, 2] bottoms = [1, 1] | sai -> 1 | đúng -> 0


Vì sao sai: có thể hàng dưới toàn `v` chỉ cần ít lần xoay hơn nhiều. **Đúng:** lấy `min(rt, rb)`.

### Lỗi 2 — Quên kiểm tra khả thi

Chỉ đếm số quân có `tops[i] != v` mà không kiểm tra `bottoms[i] == v` sẽ cho đáp án khi thực tế không thể làm.

In [22]:
tops, bottoms = counters["quên kiểm tra khả thi"]
print("tops =", tops, "bottoms =", bottoms, "| sai ->", wrong_no_feasibility(list(tops), list(bottoms)), "| đúng ->", Solution().minDominoRotations(tops, bottoms))
assert wrong_no_feasibility(list(tops), list(bottoms)) != Solution().minDominoRotations(tops, bottoms)
print("ví dụ 2: sai ->", wrong_no_feasibility(*examples[1][:2]), "(đúng -1)")

tops = [1, 2] bottoms = [1, 2] | sai -> 1 | đúng -> -1
ví dụ 2: sai -> 2 (đúng -1)


Vì sao sai: xoay quân không chứa `v` không đưa được `v` lên. **Đúng:** nếu có quân với `tops[i] != v` và `bottoms[i] != v` thì `v` không khả thi.

### Lỗi 3 — Chỉ thử ứng viên `tops[0]`

Giá trị đích có thể là `bottoms[0]` (nằm ở nửa dưới của quân đầu).

In [23]:
tops, bottoms = counters["chỉ thử ứng viên tops[0]"]
print("tops =", tops, "bottoms =", bottoms, "| sai ->", wrong_first_candidate_only(list(tops), list(bottoms)), "| đúng ->", Solution().minDominoRotations(tops, bottoms))
assert wrong_first_candidate_only(list(tops), list(bottoms)) != Solution().minDominoRotations(tops, bottoms)

tops = [1, 2] bottoms = [2, 2] | sai -> -1 | đúng -> 0


Vì sao sai: quân đầu `(a, b)` cho hai ứng viên `a` và `b`. **Đúng:** thử cả `tops[0]` và `bottoms[0]`.

### Lỗi 4 — Đếm sai quân có hai nửa cùng giá trị

Quân `(v, v)` không cần xoay cho cả hai hàng; đếm nó như cần xoay làm tăng đáp án.

In [24]:
tops, bottoms = counters["đếm sai quân hai nửa cùng giá trị"]
print("tops =", tops, "bottoms =", bottoms, "| sai ->", wrong_double_count(list(tops), list(bottoms)), "| đúng ->", Solution().minDominoRotations(tops, bottoms))
assert wrong_double_count(list(tops), list(bottoms)) != Solution().minDominoRotations(tops, bottoms)

tops = [1, 1] bottoms = [1, 2] | sai -> 1 | đúng -> 0


Vì sao sai: xoay `(v, v)` không đổi gì. **Đúng:** chỉ xoay quân có `tops[i] != v` (cho hàng trên) hoặc `bottoms[i] != v` (cho hàng dưới).

### Lỗi 5 — Trả về `0` hoặc số vô cực thay vì `-1`

Khi không khả thi đề yêu cầu `-1`.

In [25]:
tops, bottoms = counters["trả 0 thay vì -1"]
print("tops =", tops, "bottoms =", bottoms, "| sai ->", wrong_zero_when_impossible(list(tops), list(bottoms)), "| đúng ->", Solution().minDominoRotations(tops, bottoms))
assert wrong_zero_when_impossible(list(tops), list(bottoms)) != Solution().minDominoRotations(tops, bottoms)

tops = [1, 2] bottoms = [1, 2] | sai -> 0 | đúng -> -1


Vì sao sai: `0` nghĩa là "không cần xoay" (đã đồng nhất), khác "không thể". **Đúng:** trả `-1` khi cả hai ứng viên không khả thi.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chọn một giá trị chung cho mọi phần tử, mỗi phần tử có hai lựa chọn" ⇒ giá trị chung phải nằm trong **tập lựa chọn của phần tử đầu**: chỉ thử vài ứng viên.
- Khi cần kiểm "mọi phần tử chứa `v`": dùng **bù trừ** (`ct + cb - both == n`) hoặc kiểm từng phần tử.
- Với mỗi ứng viên, số thao tác thường **xác định duy nhất** (không cần tìm kiếm).
- Lấy `min` giữa các lựa chọn, `-1` khi không có lựa chọn khả thi (đừng trả `0`).
- Cùng họ: Majority Element (169), Minimum Swaps To Make Sequences Increasing (801).
- Test nên có: `n = 2`, mọi quân giống nhau, không có giá trị chung, hai ứng viên đều khả thi với số xoay khác nhau, `n = 2 * 10^4`.

## 15. Biến thể

1. **Giá trị tùy ý** (không chỉ `1..6`): dùng `Counter`/`dict` thay mảng `7`; cách hai ứng viên không đổi.
2. **Làm cả hai hàng bằng nhau** (mọi giá trị ở hai hàng đều giống nhau): cần mọi quân dạng `(v, v)` sau xoay: điều kiện `tops[i] == bottoms[i]`.
3. **In ra danh sách chỉ số cần xoay.**
4. Bài liên quan: 801, 169.

Biến thể 1 (giá trị tùy ý) kiểm bằng oracle vét cạn:

In [26]:
def domino_any_values(tops: List[int], bottoms: List[int]) -> int:
    n = len(tops)
    best = -1
    for v in {tops[0], bottoms[0]}:
        rt = rb = 0
        ok = True
        for t, b in zip(tops, bottoms):
            if t != v and b != v:
                ok = False
                break
            if t != v:
                rt += 1
            elif b != v:
                rb += 1
        if ok:
            c = min(rt, rb)
            if best == -1 or c < best:
                best = c
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 10)
    vmax = rng.choice([2, 3, 100, 10 ** 9])
    tops = [rng.randint(1, vmax) for _ in range(n)]
    bottoms = [rng.randint(1, vmax) for _ in range(n)]
    assert domino_any_values(tops, bottoms) == domino_brute(tops, bottoms)
print("biến thể giá trị tùy ý (tới 10^9): khớp oracle")

biến thể giá trị tùy ý (tới 10^9): khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập quân xoay | Theo định nghĩa | `O(2^n * n)` | `O(n)` | Oracle, `n <= 16` |
| 2. Thử 6 giá trị | Giá trị đích bất kỳ | `O(6n)` | `O(1)` | Dễ viết |
| 3. Thử 2 ứng viên | Chỉ `tops[0]`, `bottoms[0]` | `O(2n)` | `O(1)` | **Cách nộp chính** |
| 4. Đếm tần suất | Bù trừ một lượt | `O(n)` | `O(1)` | Một lượt duyệt |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(rotations_for([1, 2, 1], [2, 1, 3], 1), rotations_for([1, 2, 1], [2, 1, 3], 2))
print(Solution().minDominoRotations([1, 2], [2, 1]), Solution().minDominoRotations([1, 2], [3, 4]))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # hai số
my_guess_2 = None      # hai số

### Bài 2 — Tính tay (dễ-vừa)

`tops = [1, 2, 1, 1]`, `bottoms = [2, 1, 2, 2]`. Ghi đáp án.

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách hai ứng viên.

In [29]:
def exercise_3(tops: List[int], bottoms: List[int]) -> int:
    def check(v: int) -> int:
        rt = rb = 0
        for t, b in zip(tops, bottoms):
            # TODO 1: nếu cả hai nửa đều khác v thì trả về -1
            # TODO 2: nếu t != v thì rt += 1; ngược lại nếu b != v thì rb += 1
            pass
        return min(rt, rb)

    # TODO 3: thử v = tops[0] và v = bottoms[0]; trả về nhỏ nhất trong các kết quả khác -1, hoặc -1
    pass


# for tops, bottoms, expected in examples:
#     assert exercise_3(tops, bottoms) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(tops, bottoms)` bằng **đếm tần suất** (`ct`, `cb`, `both`).

In [30]:
def exercise_4(tops: List[int], bottoms: List[int]) -> int:
    # TODO
    pass


# for tops, bottoms, expected in examples:
#     assert exercise_4(tops, bottoms) == expected

### Bài 5 — Mini challenge: những giá trị đích khả thi (khó)

Viết `exercise_5(tops, bottoms)` trả về **danh sách tăng dần** các giá trị `v` sao cho **có thể** làm hàng trên hoặc hàng dưới toàn `v` (không quan tâm số lần xoay).

```python
exercise_5([2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2]) -> [2]
exercise_5([3, 5, 1, 2, 3], [3, 6, 3, 3, 4])       -> []
exercise_5([1, 2], [2, 1])                         -> [1, 2]
exercise_5([1, 1], [1, 1])                         -> [1]
```
Gợi ý: `v` khả thi ⇔ mọi quân chứa `v`; chỉ cần thử `tops[0]` và `bottoms[0]`.

In [31]:
def exercise_5(tops: List[int], bottoms: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_5([2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2]) == [2]
# assert exercise_5([3, 5, 1, 2, 3], [3, 6, 3, 3, 4]) == []
# assert exercise_5([1, 2], [2, 1]) == [1, 2]
# assert exercise_5([1, 1], [1, 1]) == [1]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `rotations_for` trả `-1` nếu có quân không chứa `v`.
- Mức 2: `[1,2,1]` / `[2,1,3]`: quân thứ ba `(1, 3)` chứa `1` nhưng không chứa `2`.
- Mức 3: dòng 1: `1 -1`; dòng 2: `1 -1`.

**Bài 2**
- Mức 1: các ứng viên là `1` và `2` (quân đầu `(1, 2)`).
- Mức 2: với `v = 1`: hàng trên cần xoay quân chỉ số `1` (1 lần); hàng dưới cần xoay 3 lần. Với `v = 2`: hàng trên cần xoay 3 lần; hàng dưới cần xoay 1 lần.
- Mức 3: đáp án `1`.

**Bài 3**
- Mức 1: `if t != v and b != v: return -1`.
- Mức 2: `if t != v: rt += 1` `elif b != v: rb += 1`.
- Mức 3: gom `[r for r in (check(tops[0]), check(bottoms[0])) if r != -1]`, trả `min(...)` hoặc `-1`.

**Bài 4**
- Mức 1: `ct[t] += 1`, `cb[b] += 1`, và `both[t] += 1` nếu `t == b`.
- Mức 2: `v` khả thi nếu `ct[v] + cb[v] - both[v] == n`.
- Mức 3: số xoay `n - max(ct[v], cb[v])`, lấy nhỏ nhất.

**Bài 5**
- Mức 1: chỉ ứng viên `tops[0]`, `bottoms[0]`.
- Mức 2: với mỗi ứng viên kiểm `all(t == v or b == v for ...)`.
- Mức 3: trả `sorted(...)` các ứng viên khả thi (loại trùng).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(rotations_for([1, 2, 1], [2, 1, 3], 1), rotations_for([1, 2, 1], [2, 1, 3], 2))
print(Solution().minDominoRotations([1, 2], [2, 1]), Solution().minDominoRotations([1, 2], [3, 4]))

assert (rotations_for([1, 2, 1], [2, 1, 3], 1), rotations_for([1, 2, 1], [2, 1, 3], 2)) == (1, -1)
assert (Solution().minDominoRotations([1, 2], [2, 1]), Solution().minDominoRotations([1, 2], [3, 4])) == (1, -1)

1 -1
1 -1


### Đáp án Bài 2

`v = 1`: hàng trên thiếu `1` ở chỉ số `1` (quân `(2, 1)` chứa `1`) nên cần `1` lần xoay. `v = 2`: hàng dưới thiếu `2` ở chỉ số `1` nên cần `1` lần. Nhỏ nhất `1`.

In [33]:
hand_answer_solution = 1
assert Solution().minDominoRotations([1, 2, 1, 1], [2, 1, 2, 2]) == hand_answer_solution == domino_brute([1, 2, 1, 1], [2, 1, 2, 2])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(tops: List[int], bottoms: List[int]) -> int:
    def check(v: int) -> int:
        rt = rb = 0
        for t, b in zip(tops, bottoms):
            if t != v and b != v:
                return -1
            if t != v:
                rt += 1
            elif b != v:
                rb += 1
        return min(rt, rb)

    ans = [r for r in (check(tops[0]), check(bottoms[0])) if r != -1]
    return min(ans) if ans else -1

for tops, bottoms, expected in examples:
    assert exercise_3_solution(tops, bottoms) == expected
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(2, 10)
    tops = [rng.randint(1, 3) for _ in range(n)]
    bottoms = [rng.randint(1, 3) for _ in range(n)]
    assert exercise_3_solution(tops, bottoms) == domino_brute(tops, bottoms)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(tops: List[int], bottoms: List[int]) -> int:
    n = len(tops)
    ct, cb, both = [0] * 7, [0] * 7, [0] * 7
    for t, b in zip(tops, bottoms):
        ct[t] += 1
        cb[b] += 1
        if t == b:
            both[t] += 1
    best = -1
    for v in range(1, 7):
        if ct[v] + cb[v] - both[v] == n:
            r = n - max(ct[v], cb[v])
            if best == -1 or r < best:
                best = r
    return best

for tops, bottoms, expected in examples:
    assert exercise_4_solution(tops, bottoms) == expected
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 10)
    tops = [rng.randint(1, 3) for _ in range(n)]
    bottoms = [rng.randint(1, 3) for _ in range(n)]
    assert exercise_4_solution(tops, bottoms) == domino_brute(tops, bottoms)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(tops: List[int], bottoms: List[int]) -> List[int]:
    out = []
    for v in sorted({tops[0], bottoms[0]}):
        if all(t == v or b == v for t, b in zip(tops, bottoms)):
            out.append(v)
    return out

assert exercise_5_solution([2, 1, 2, 4, 2, 2], [5, 2, 6, 2, 3, 2]) == [2]
assert exercise_5_solution([3, 5, 1, 2, 3], [3, 6, 3, 3, 4]) == []
assert exercise_5_solution([1, 2], [2, 1]) == [1, 2]
assert exercise_5_solution([1, 1], [1, 1]) == [1]
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(2, 8)
    tops = [rng.randint(1, 4) for _ in range(n)]
    bottoms = [rng.randint(1, 4) for _ in range(n)]
    expect = [v for v in range(1, 7) if all(t == v or b == v for t, b in zip(tops, bottoms))]
    assert exercise_5_solution(tops, bottoms) == expect
    assert (domino_brute(tops, bottoms) != -1) == bool(expect)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (hai ứng viên)

```python
class Solution:
    def minDominoRotations(self, tops: List[int], bottoms: List[int]) -> int:
        def check(v: int) -> int:
            rt = rb = 0
            for t, b in zip(tops, bottoms):
                if t != v and b != v:
                    return -1                  # có quân không chứa v
                if t != v:
                    rt += 1                    # xoay để đưa v lên hàng trên
                elif b != v:
                    rb += 1                    # xoay để đưa v xuống hàng dưới
            return min(rt, rb)

        ans = [r for r in (check(tops[0]), check(bottoms[0])) if r != -1]
        return min(ans) if ans else -1
```

Công thức nhớ nhanh: giá trị đích ∈ `{tops[0], bottoms[0]}`; số xoay `= min(#quân thiếu v ở hàng trên, #quân thiếu v ở hàng dưới)`.

### Checklist trước khi submit

- [ ] Thử **cả hai** ứng viên `tops[0]` và `bottoms[0]`?
- [ ] Kiểm tra mọi quân chứa `v` ở ít nhất một nửa (không chứa thì `-1` cho `v` đó)?
- [ ] Lấy `min` của số xoay cho **hàng trên** và **hàng dưới**?
- [ ] Quân `(v, v)` không bị tính là phải xoay?
- [ ] Trả `-1` (không phải `0`) khi cả hai ứng viên không khả thi?
- [ ] Đã thử: `n = 2`, mọi quân giống nhau, không có giá trị chung, hai ứng viên cùng khả thi, `n = 2 * 10^4`?